In [0]:
%run ../bronze/bronze_dq_checks

In [0]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

In [0]:
from delta.tables import DeltaTable 
from pyspark.sql import functions as F

In [0]:
# Load the shared silver functions (transform, filter_new_rows, ...) from the
# 'silver' notebook by executing its Python source in this session.
# (%run cannot be used here: a %run cell typed in the UI works, but this
# notebook's execution path needs a Python cell.)
import base64
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat

_silver_source = base64.b64decode(
    WorkspaceClient().workspace.export(
        path="/Users/trainings@quenrix.com/usdeadb-ecomm/src/silver/silver",
        format=ExportFormat.SOURCE,
    ).content
).decode()

# Skip magic lines (the bronze %run inside silver) and execute only plain Python
exec(
    compile(
        "\n".join(l for l in _silver_source.splitlines() if not l.startswith("# MAGIC")),
        "silver_notebook",
        "exec",
    ),
    globals(),
)
del _silver_source

print("Loaded from silver:", "filter_new_rows" in globals(), "transform" in globals())

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "full"])

catalog = dbutils.widgets.get("catalog")
mode = dbutils.widgets.get("mode")

In [0]:
bronze_table = f"{catalog}.bronze.raw_files"
silver_orders = f"{catalog}.silver.orders"
silver_rejected = f"{catalog}.silver.orders_rejected"

In [0]:
# spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

In [0]:
def get_watermark():
    if mode == "full":
        return None
    marks = []
    for t in (silver_orders, silver_rejected):
        if spark.catalog.tableExists(t):
            marks.append(spark.table(t).agg(F.max("_ingestTimeStamp")).collect()[0][0])
    marks = [m for m in marks if m is not None]
    return max(marks) if marks else None


watermark = get_watermark()
print(f"mode = {mode}, watermark = {watermark}")

new_rows = filter_new_rows(spark.table(bronze_table), watermark)

if new_rows.isEmpty():
    dbutils.notebook.exit("No New Bronze Data")
    


In [0]:
valid_df, rejected_df = transform(new_rows)
valid_df = valid_df.cache()
rejected_df = rejected_df.cache()

print(f"bronze rows read: {new_rows.count()}")
print(f"valid_orders: {valid_df.count()}")
print(f"rejected rows : {rejected_df.count()}")


In [0]:
def upsert(df, target, merge_condition, update_condition=None):
    if not spark.catalog.tableExists(target):
        df.write.format("delta").saveAsTable(target)
        return
        (DeltaTable.forName(spark, target).alias("t")
         .merge(df.alias("s"), merge_condition)
         .whenMatechedUpdateAll(condition=update_condition)
         .whenNotMatchedInsertAll()
         .execute())
        

upsert(valid_df, silver_orders, "t.order_id=s.order_id", "s._ingestTimeStamp >= t._ingestTimeStamp")

upsert(rejected_df, silver_rejected, "t.bronze_record_id= s.bronze_record_id")



In [0]:
print("----- silver-orders------")
display(spark.table(silver_orders).orderBy("order_id"))

print("------silver.orders_rejected-------")
display(spark.table(silver_rejected).select("bronze_record_id", "order_id", "raw_order_date", "quantity", "unit_price", "reject_reason"))


dup_orders = (spark.table(silver_orders).groupBy("order_id").count().where("count > 1").count())
assert dup_orders == 0, f"silver.orders has {dup_orders} duplicated order_id values"
print("Silver Checks Passed")